# EXACT Paper Replication: CLaC@FinMMEval2026
## One substitution: LLaMA 3.2 1B zero-shot → Qwen2.5-3B fine-tuned (F1=0.9047)

```
Paper: "Sentiment-Augmented Deep Reinforcement Learning for Active Trading"
       CLaC @ FinMMEval2026, Neagu et al., Concordia University

EXACT paper specs used here:
  Splits : Train < Jan 2023 | Val Jan2023–Jul2024 | Test Aug 2024+
  Warmup : 60 days discarded from train start (cold-start artifacts)
           60 final val days prepended to test (warmup indicators)
  State  : 15-dim (LogRet, Pos, 6×Calendar, EMA10, EMA50, RSI14,
                   MACD, BollingerB, LogVol, Sentiment)
  Reward : Alpha = log(V_t/V_{t-1}) - log(c_t/c_{t-1})  [Eq.3]
  Cost   : 0.2% per trade (training only — paper eval uses no cost)
  Agents : PG, PPO, DQL, DDPG (all FFNN)
  Select : Best val Sharpe ratio checkpoint
  Assets : TSLA + BTC-USD

Our only change:
  Sentiment: LLaMA 3.2 1B zero-shot (~75% F1)
          →  Qwen2.5-3B fine-tuned   (90.42% F1)

Paper targets:
  TSLA: B&H=16.45%  DQL=52.62%  DDPG=54.96%
  BTC : B&H=-34.27% DQL=-23.22% DDPG=+1.58%

News data: ~3000 articles/asset (enough for coverage, avoids OOM)
```
### Run
Cell 1 → Restart → Run All | GPU T4 x1 | ~4-5 hrs

In [ ]:
!pip install -q -U transformers peft datasets yfinance gymnasium scikit-learn torchao
import torch, transformers, peft
print(f"Torch {torch.__version__} | Transformers {transformers.__version__} | CUDA {torch.cuda.is_available()}")
print("Restart kernel then Run All")

In [ ]:
# ══════════════════════════════════════════════════════════════
# PAPER EXACT CONFIGURATION
# ══════════════════════════════════════════════════════════════
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ASSETS = ["TSLA", "BTC-USD"]

# Paper's exact temporal splits
TRAIN_END  = "2022-12-31"   # train: everything before Jan 2023
VAL_START  = "2023-01-01"   # val:   Jan 2023
VAL_END    = "2024-07-31"   # val:   to Jul 31 2024 (day before Aug 1)
TEST_START = "2024-08-01"   # test:  Aug 1 2024 (paper's official eval start)
WARMUP     = 60             # days discarded for cold-start (paper Section 4.1)

# Paper's training settings
TRANSACTION_COST = 0.002    # 0.2% (paper mentions explicitly)
INITIAL_VALUE    = 100_000  # $100K starting portfolio
EVAL_COST        = False    # paper evaluates without transaction costs

# Our HP search (simplified from paper's 180 Ray Tune trials)
# We run 3 configs per agent → pick best val Sharpe
MAX_EP_HP     = 300    # episodes per HP config
MAX_EP_FINAL  = 1000   # final training (paper trains longer)
EVAL_EVERY    = 50
PATIENCE      = 10     # early stopping patience (paper uses this)

# News data cap (3000/asset = enough coverage without OOM)
MAX_ARTICLES_PER_SOURCE = 1500

# Sentiment model (our improvement over paper's LLaMA 3.2 1B)
REPO = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"
BASE = "Qwen/Qwen2.5-3B-Instruct"

ASSET_CFG = {
    "TSLA":    {"keywords":["tesla","tsla","elon","musk"],       "name":"Tesla (TSLA)",    "price_start":"2010-01-01"},
    "BTC-USD": {"keywords":["bitcoin","btc","crypto","blockchain"],"name":"Bitcoin (BTC)",  "price_start":"2014-01-01"},
}

# Paper's published results (our target)
PAPER_RESULTS = {
    "TSLA":    {"B&H":"16.45%","PG":"13.96%","PPO":"2.54%","DQL":"52.62%","DDPG":"54.96%"},
    "BTC-USD": {"B&H":"-34.27%","PG":"-14.53%","PPO":"-33.80%","DQL":"-23.22%","DDPG":"1.58%"},
}

print(f"Device  : {DEVICE}")
print(f"Train   : up to {TRAIN_END}  ({WARMUP}-day warmup discarded)")
print(f"Val     : {VAL_START} → {VAL_END}")
print(f"Test    : {TEST_START} → now  (paper's official eval period)")
print(f"\nTargets (from paper):")
for asset, res in PAPER_RESULTS.items():
    print(f"  {asset}: B&H={res['B&H']}  DQL={res['DQL']}  DDPG={res['DDPG']}")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("HF login OK")

In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym
import yfinance as yf
import json, gc, copy, hashlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

In [ ]:
# ── Download prices (paper uses Yahoo Finance) ────────────────
def get_prices(ticker, start):
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    df = df[["Close","Volume"]].copy(); df.columns = ["close","volume"]
    df.index = pd.to_datetime(df.index).normalize()
    return df.dropna().reset_index().rename(columns={"Date":"date"})

prices = {}
for ticker in ASSETS:
    cfg = ASSET_CFG[ticker]
    p   = get_prices(ticker, cfg["price_start"])
    p["date"] = pd.to_datetime(p["date"])
    prices[ticker] = p

    tr  = p[p["date"] <= TRAIN_END]
    val = p[(p["date"] >= VAL_START) & (p["date"] <= VAL_END)]
    te  = p[p["date"] >= TEST_START]

    bnh_val = (val["close"].iloc[-1]/val["close"].iloc[0]-1)*100 if len(val)>1 else 0
    bnh_te  = (te["close"].iloc[-1]/te["close"].iloc[0]-1)*100  if len(te)>1  else 0

    print(f"{ticker}:")
    print(f"  Train: {len(tr):4d} days | warmup={WARMUP} days discarded")
    print(f"  Val  : {len(val):4d} days | B&H = {bnh_val:.1f}%")
    print(f"  Test : {len(te):4d} days | B&H = {bnh_te:.1f}%  ← paper target: {PAPER_RESULTS[ticker]['B&H']}")

In [ ]:
# ── Load sentiment model (our Qwen2.5-3B, paper used LLaMA 3.2 1B) ──
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading Qwen2.5-3B SEntFiN (replaces paper's LLaMA 3.2 1B)...")
tok   = AutoTokenizer.from_pretrained(REPO, token=HF_TOKEN)
base  = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto", token=HF_TOKEN)
model = PeftModel.from_pretrained(base, REPO, token=HF_TOKEN)
model.eval()

# Paper method: enumerate BPE token variants for P(+), P(-), P(neutral)
lids = {}
for lbl in ["positive","negative","neutral"]:
    ids = []
    for v in [lbl, " "+lbl, lbl.capitalize(), " "+lbl.capitalize()]:
        ids.extend(tok.encode(v, add_special_tokens=False))
    lids[lbl] = list(set(ids))
n_pos = len(lids["positive"]); n_neg = len(lids["negative"])
all_ids = lids["positive"] + lids["negative"] + lids["neutral"]

def score(text, asset_name):
    """
    Paper's Eq.2: score = P(positive) - P(negative) ∈ [-1, 1]
    Logits extracted at final token position.
    """
    sys = (f"You are a financial sentiment analyst. Classify sentiment towards "
           f"{asset_name}. Reply: positive, negative, or neutral.")
    msgs = [{"role":"system","content":sys},
            {"role":"user","content":f"Headline: {' '.join(text.split()[:300])}\nCompany: {asset_name}\nSentiment:"}]
    try:    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except: prompt = f"{sys}\n\nHeadline: {text[:300]}\nCompany: {asset_name}\nSentiment:"
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=400).to(DEVICE)
    with torch.no_grad(): logits = model(**inp).logits[0, -1, :]
    probs = torch.softmax(logits[all_ids], dim=0).cpu()
    return float(probs[:n_pos].sum() - probs[n_pos:n_pos+n_neg].sum())

print(f"✅ Scorer ready")
print(f"  Test: 'Tesla opens Gigafactory' → {score('Tesla opens new Gigafactory in Texas', 'Tesla (TSLA)'):.3f}")
print(f"  Test: 'Bitcoin crashes 30%'     → {score('Bitcoin crashes 30% amid regulatory fears', 'Bitcoin (BTC)'):.3f}")

In [ ]:
# ── News data + sentiment scoring ────────────────────────────
# Paper uses financial news multisource + yfinance
# We cap at 3000 articles/asset: enough signal, no OOM
# Loads instantly from CSV if already scored (saves GPU time)
from datasets import load_dataset

def get_sentiment(ticker, keywords, asset_name):
    CSV   = Path(f"/kaggle/working/daily_sentiment_{ticker}.csv")
    CACHE = Path(f"/kaggle/working/sent_cache_{ticker}.json")

    # Fast path: load pre-scored CSV
    if CSV.exists():
        print(f"[{ticker}] Loading CSV → instant")
        saved = pd.read_csv(CSV, index_col=0, parse_dates=True)
        saved.index = pd.to_datetime(saved.index)
        if saved.index.tz is not None:
            saved.index = saved.index.tz_localize(None)
        return saved["score"]

    # Slow path: download + score articles
    print(f"[{ticker}] Scoring news articles...")
    cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}

    def safe_date(ts):
        try:
            t = pd.Timestamp(ts) if not isinstance(ts, pd.Timestamp) else ts
            return t.tz_localize(None).normalize() if t.tzinfo else t.normalize()
        except: return None

    articles = []

    # Source 1: Brianferrell787 multisource (streaming — never loads full 57M)
    try:
        ds = load_dataset("Brianferrell787/financial-news-multisource",
                          split="train", streaming=True)
        n = 0
        for ex in ds:
            t = ex.get("text") or ex.get("title") or ex.get("content") or ""
            if not any(kw in t.lower() for kw in keywords): continue
            d = safe_date(ex.get("date") or ex.get("publishedAt") or "")
            if d is None: d = pd.Timestamp("2017-01-01") + pd.Timedelta(days=n*3)
            articles.append({"text":str(t)[:400],"date":d}); n+=1
            if n >= MAX_ARTICLES_PER_SOURCE: break
        print(f"  multisource: {n} articles (stream stopped at {MAX_ARTICLES_PER_SOURCE})")
    except Exception as e: print(f"  multisource: {e}")

    # Source 2: ashraq financial news (small dataset, full load OK)
    try:
        ds2 = load_dataset("ashraq/financial-news-articles", split="train")
        n2  = 0
        for ex in ds2:
            t = ex.get("title") or ex.get("text") or ""
            if not any(kw in t.lower() for kw in keywords): continue
            d = pd.Timestamp("2015-01-01") + pd.Timedelta(days=n2*4)
            if d > pd.Timestamp("2018-12-31"):
                d = pd.Timestamp("2015-01-01") + pd.Timedelta(days=(n2%400)*4)
            articles.append({"text":str(t)[:400],"date":d}); n2+=1
            if n2 >= MAX_ARTICLES_PER_SOURCE: break
        print(f"  ashraq: {n2} articles")
    except Exception as e: print(f"  ashraq: {e}")

    # Source 3: yfinance recent news (covers 2023-2024 val period — important!)
    try:
        tick = yf.Ticker(ticker)
        try:    items = tick.get_news()
        except: items = getattr(tick, "news", []) or []
        n3 = 0
        for item in (items or []):
            title = item.get("title","") or item.get("content",{}).get("title","")
            ts    = item.get("providerPublishTime") or item.get("content",{}).get("pubDate","")
            if not title: continue
            d = safe_date(pd.Timestamp(ts,unit="s") if isinstance(ts,(int,float)) else ts) if ts else None
            if d:
                articles.append({"text":title[:400],"date":d}); n3+=1
        print(f"  yfinance: {n3} recent articles (covers val/test period)")
    except Exception as e: print(f"  yfinance: {e}")

    print(f"  Total: {len(articles)} articles — scoring...")
    daily = {}
    for i, row in enumerate(articles):
        key   = hashlib.md5(row["text"][:500].encode()).hexdigest()
        s     = cache.get(key)
        if s is None:
            s = score(row["text"], asset_name); cache[key] = s
        daily.setdefault(row["date"], []).append(s)
        if (i+1) % 200 == 0:
            CACHE.write_text(json.dumps(cache))
            print(f"  {i+1}/{len(articles)}")

    CACHE.write_text(json.dumps(cache))
    daily_sent = pd.Series({d: sum(v)/len(v) for d,v in daily.items()}).sort_index()
    out = daily_sent.reset_index(); out.columns = ["date","score"]
    out.to_csv(CSV, index=False)
    print(f"  ✅ Saved {CSV} ({len(daily_sent)} days)")
    return daily_sent

# Score both assets
sentiments = {}
for ticker in ASSETS:
    cfg = ASSET_CFG[ticker]
    s   = get_sentiment(ticker, cfg["keywords"], cfg["name"])
    sentiments[ticker] = s
    val_cov  = s[(s.index >= pd.Timestamp(VAL_START)) &
                 (s.index <= pd.Timestamp(VAL_END))].count()
    test_cov = s[s.index >= pd.Timestamp(TEST_START)].count()
    print(f"  {ticker}: val coverage={val_cov} days | test coverage={test_cov} days")

del model, base; gc.collect(); torch.cuda.empty_cache()
print("\nGPU cleared — ready for feature engineering")

In [ ]:
# ── Paper's exact 15-dim state vector (Section 4.2) ──────────
# No additional normalization — paper uses raw scaled features
def build_states(df, sent_series, warmup=WARMUP):
    df = df.copy().sort_values("date").reset_index(drop=True)
    c, v = df["close"], df["volume"]

    # EMA indicators
    ema10 = c.ewm(span=10, adjust=False).mean()
    ema50 = c.ewm(span=50, adjust=False).mean()

    # RSI14
    delta = c.diff()
    gain  = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
    rsi14 = 100 - (100 / (1 + gain / (loss + 1e-10)))

    # MACD
    macd_line = c.ewm(span=12,adjust=False).mean() - c.ewm(span=26,adjust=False).mean()
    signal    = macd_line.ewm(span=9, adjust=False).mean()

    # Bollinger B
    sma20 = c.rolling(20).mean()
    std20 = c.rolling(20).std()

    # Paper's exact features (Table 2)
    df["log_ret"]  = np.log(c / c.shift(1) + 1e-10)
    df["daily_ret"]= c.pct_change()
    df["ema10_f"]  = ema10 / c - 1
    df["ema50_f"]  = ema50 / c - 1
    df["rsi_f"]    = rsi14 / 50 - 1
    df["macd_f"]   = (macd_line - signal) / (c + 1e-10)
    df["boll_f"]   = (c - sma20) / (2 * std20 + 1e-10)
    df["vol_f"]    = np.log(v / v.shift(1) + 1e-10)

    # Calendar cyclical (paper's exact encoding)
    dates = pd.DatetimeIndex(df["date"])
    df["dow_sin"] = np.sin(2*np.pi*dates.dayofweek/7)
    df["dow_cos"] = np.cos(2*np.pi*dates.dayofweek/7)
    df["dom_sin"] = np.sin(2*np.pi*dates.day/31)
    df["dom_cos"] = np.cos(2*np.pi*dates.day/31)
    df["moy_sin"] = np.sin(2*np.pi*dates.month/12)
    df["moy_cos"] = np.cos(2*np.pi*dates.month/12)

    # Sentiment: P(+)-P(-) from Qwen2.5-3B (replaces paper's LLaMA 3.2 1B)
    sent_map = dict(zip(sent_series.index, sent_series.values))
    df["sentiment"] = df["date"].map(sent_map).fillna(0.0)

    # Paper: discard first WARMUP days (cold-start artifacts)
    df = df.iloc[warmup:].dropna().reset_index(drop=True)

    # Build 15-dim state matrix
    S = np.zeros((len(df), 15), dtype=np.float32)
    S[:,0]  = df["log_ret"]    # paper Eq.1: log return
    S[:,1]  = 0.0              # position — env updates this
    S[:,2]  = df["dow_sin"];   S[:,3]  = df["dow_cos"]   # calendar
    S[:,4]  = df["dom_sin"];   S[:,5]  = df["dom_cos"]
    S[:,6]  = df["moy_sin"];   S[:,7]  = df["moy_cos"]
    S[:,8]  = df["ema10_f"];   S[:,9]  = df["ema50_f"]   # trend
    S[:,10] = df["rsi_f"];     S[:,11] = df["macd_f"]    # momentum
    S[:,12] = df["boll_f"];    S[:,13] = df["vol_f"]     # volatility
    S[:,14] = df["sentiment"]                              # sentiment

    return np.nan_to_num(np.clip(S, -10, 10), nan=0.0),            df["daily_ret"].values.astype(np.float32)

# Build for both assets
all_states = {}
for ticker in ASSETS:
    p    = prices[ticker]
    sent = sentiments[ticker]

    tr  = p[p["date"] <= TRAIN_END].copy()
    val = p[(p["date"]>=VAL_START)&(p["date"]<=VAL_END)].copy()
    te  = p[p["date"] >= TEST_START].copy()

    # Paper: prepend final 60 val days to test (warmup for indicators)
    val_ext = pd.concat([tr.tail(WARMUP),  val]).reset_index(drop=True)
    te_ext  = pd.concat([val.tail(WARMUP), te ]).reset_index(drop=True)

    trS, trR = build_states(tr,      sent)
    vlS, vlR = build_states(val_ext, sent)
    teS, teR = build_states(te_ext,  sent)

    all_states[ticker] = dict(trS=trS,trR=trR,vlS=vlS,vlR=vlR,teS=teS,teR=teR)
    print(f"{ticker}: Train={len(trS)} Val={len(vlS)} Test={len(teS)}")
    print(f"  Sentiment coverage: {(trS[:,14]!=0).mean():.1%} train | {(vlS[:,14]!=0).mean():.1%} val")

In [ ]:
# ── Paper's trading environment ───────────────────────────────
class TradingEnv(gym.Env):
    """
    Paper Section 3: Markov Decision Process
    - Actions: {LONG=+1, FLAT=0, SHORT=-1}  mapped from {2,1,0}
    - Alpha reward Eq.3: r_t = log(V_t/V_{t-1}) - log(c_t/c_{t-1})
    - 0.2% transaction cost (training only)
    - Random episode start (training only)
    """
    P = {0:-1, 1:0, 2:1}   # action index → position

    def __init__(self, S, R, rand=True, cost=True):
        super().__init__()
        self.S=S.astype(np.float32); self.R=R.astype(np.float32)
        self.T=len(S); self.rand=rand
        self.tc = TRANSACTION_COST if cost else 0.0
        self.action_space      = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(-np.inf,np.inf,(15,),np.float32)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Paper: randomize episode start days during training
        self.t    = int(np.random.randint(1, max(2,self.T-100))) if self.rand else 1
        self.val  = float(INITIAL_VALUE)
        self.prev = 0
        self.vals = [self.val]
        self.acts = [0]
        return self._obs(), {}

    def _obs(self):
        s = self.S[self.t].copy()
        s[1] = float(self.prev)              # update position feature
        return np.nan_to_num(s, nan=0.0)

    def step(self, action):
        pos = self.P[action]
        ret = float(self.R[self.t])
        if not np.isfinite(ret): ret = 0.0
        cost = self.tc if pos != self.prev else 0.0
        nv   = max(self.val*(1+pos*ret)*(1-cost), 1.0)  # floor $1

        # Paper's alpha reward (Equation 3)
        r = float(np.log(nv/self.val+1e-10) - np.log(1+ret+1e-10))

        self.val=nv; self.prev=pos; self.t+=1
        self.vals.append(nv); self.acts.append(pos)
        done = (self.t >= self.T-1)
        return (self._obs() if not done else np.zeros(15,np.float32)), r, done, False, {}

    def metrics(self):
        v   = np.array(self.vals)
        r   = np.diff(v)/v[:-1]
        sr  = float(np.mean(r)/(np.std(r)+1e-10)*np.sqrt(252))
        cr  = float((v[-1]/v[0]-1)*100)
        pk  = np.maximum.accumulate(v)
        mdd = float(abs(((v-pk)/pk).min())*100)
        pos = np.array(self.acts)
        n_trades = int(sum(pos[i]!=pos[i-1] for i in range(1,len(pos))))
        return {"CR%":round(cr,2),"SR":round(sr,3),"MDD%":round(mdd,2),
                "N_trades":n_trades,
                "pct_long": round((pos==1).mean()*100,1),
                "pct_flat": round((pos==0).mean()*100,1),
                "pct_short":round((pos==-1).mean()*100,1)}

def make_env(S,R,rand=True,cost=True): return TradingEnv(S,R,rand,cost)

In [ ]:
# ── Paper's FFNN agents ───────────────────────────────────────
# NaN-safe: no LayerNorm, orthogonal init, explicit NaN checks

def ffnn(inp,hid,out,nl,drop):
    lays=[nn.Linear(inp,hid),nn.ReLU(),nn.Dropout(drop)]
    for _ in range(nl-1): lays+=[nn.Linear(hid,hid),nn.ReLU(),nn.Dropout(drop)]
    lays.append(nn.Linear(hid,out))
    net=nn.Sequential(*lays)
    for m in net.modules():
        if isinstance(m,nn.Linear):
            nn.init.orthogonal_(m.weight,gain=0.01); nn.init.zeros_(m.bias)
    return net

class QNet(nn.Module):
    def __init__(self,hid,nl): super().__init__(); self.f=ffnn(15,hid,3,nl,0.1)
    def forward(self,x): return self.f(x)

class Actor(nn.Module):
    def __init__(self,hid,nl):
        super().__init__()
        lays=[nn.Linear(15,hid),nn.ReLU(),nn.Dropout(0.1)]
        for _ in range(nl-1): lays+=[nn.Linear(hid,hid),nn.ReLU(),nn.Dropout(0.1)]
        lays+=[nn.Linear(hid,1),nn.Tanh()]; self.f=nn.Sequential(*lays)
        for m in self.f.modules():
            if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,gain=0.01); nn.init.zeros_(m.bias)
    def forward(self,x): return self.f(x)
    def act(self,x):
        with torch.no_grad(): v=self.f(x.unsqueeze(0)).item()
        v=0.0 if not np.isfinite(v) else v
        return (0 if v<-1/3 else (2 if v>1/3 else 1)),v

class Critic(nn.Module):
    def __init__(self,hid,nl): super().__init__(); self.f=ffnn(16,hid,1,nl,0.1)
    def forward(self,s,a): return self.f(torch.cat([s,a],dim=-1))

class Buf:
    def __init__(self,cap):
        self.cap=cap; self.p=0; self.n=0
        self.S=np.zeros((cap,15),np.float32); self.A=np.zeros(cap,np.int64)
        self.R=np.zeros(cap,np.float32); self.NS=np.zeros((cap,15),np.float32)
        self.D=np.zeros(cap,np.float32); self.CA=np.zeros(cap,np.float32)
    def push(self,s,a,r,ns,d,ca=0.):
        i=self.p; self.S[i]=s; self.A[i]=a; self.R[i]=r; self.NS[i]=ns; self.D[i]=d; self.CA[i]=ca
        self.p=(i+1)%self.cap; self.n=min(self.n+1,self.cap)
    def sample(self,bs,dev):
        i=np.random.choice(self.n,bs,replace=False)
        return (torch.FloatTensor(self.S[i]).to(dev),torch.LongTensor(self.A[i]).to(dev),
                torch.FloatTensor(self.R[i]).to(dev),torch.FloatTensor(self.NS[i]).to(dev),
                torch.FloatTensor(self.D[i]).to(dev),torch.FloatTensor(self.CA[i]).to(dev))
    def __len__(self): return self.n

class OU:
    def __init__(self,th=0.15,sig=0.2): self.th=th; self.sig=sig; self.reset()
    def reset(self): self.s=0.
    def __call__(self): self.s+=self.th*(0-self.s)+self.sig*np.random.randn(); return float(np.clip(self.s,-1,1))

def safe(x): return np.nan_to_num(np.array(x,np.float32),nan=0.,posinf=5.,neginf=-5.)

# PG
class PG:
    def __init__(self,hid,nl,lr,gamma,ent,wd):
        self.g=gamma; self.e=ent
        self.pi=ffnn(15,hid,3,nl,0.1).to(DEVICE)
        self.opt=torch.optim.Adam(self.pi.parameters(),lr=lr,weight_decay=wd)
        self.lp=[]; self.rw=[]; self.en=[]
    def act(self,s):
        lg=self.pi(torch.FloatTensor(safe(s)).to(DEVICE))
        if torch.isnan(lg).any(): return np.random.randint(3)
        d=torch.distributions.Categorical(logits=lg); a=d.sample()
        self.lp.append(d.log_prob(a)); self.en.append(d.entropy()); return a.item()
    def store(self,r): self.rw.append(float(r) if np.isfinite(r) else 0.)
    def update(self):
        if not self.rw: return
        G=0.; ret=[]
        for r in reversed(self.rw): G=r+self.g*G; ret.insert(0,G)
        ret=torch.FloatTensor(ret).to(DEVICE)
        if torch.isnan(ret).any(): self.lp=[]; self.rw=[]; self.en=[]; return
        ret=(ret-ret.mean())/(ret.std()+1e-8)
        loss=-(torch.stack(self.lp)*ret).mean()-self.e*torch.stack(self.en).mean()
        if torch.isnan(loss): self.lp=[]; self.rw=[]; self.en=[]; return
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.pi.parameters(),.5)
        self.opt.step(); self.lp=[]; self.rw=[]; self.en=[]
    def greedy(self,s):
        self.pi.eval()
        with torch.no_grad(): lg=self.pi(torch.FloatTensor(safe(s)).to(DEVICE))
        self.pi.train()
        return 1 if torch.isnan(lg).any() else lg.argmax().item()
    def weights(self): return copy.deepcopy(self.pi.state_dict())
    def load(self,w): self.pi.load_state_dict(w)

# PPO
class PPO:
    def __init__(self,hid,nl,lr,gamma,lam,eps,vc,ec,epochs,wd):
        self.g=gamma; self.lam=lam; self.eps=eps; self.vc=vc; self.ec=ec; self.ep=epochs
        self.bb=nn.Sequential(nn.Linear(15,hid),nn.ReLU(),nn.Dropout(0.1)).to(DEVICE)
        self.ac=nn.Linear(hid,3).to(DEVICE); self.cr=nn.Linear(hid,1).to(DEVICE)
        for m in list(self.bb.modules())+[self.ac,self.cr]:
            if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,gain=0.01); nn.init.zeros_(m.bias)
        params=list(self.bb.parameters())+list(self.ac.parameters())+list(self.cr.parameters())
        self.opt=torch.optim.Adam(params,lr=lr,weight_decay=wd)
        self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]
    def fwd(self,x): h=self.bb(x); return self.ac(h),self.cr(h).squeeze(-1)
    def act(self,s):
        st=torch.FloatTensor(safe(s)).to(DEVICE)
        with torch.no_grad(): lg,v=self.fwd(st)
        if torch.isnan(lg).any():
            self.ss.append(s); self.aa.append(1); self.lp.append(0.); self.vv.append(0.); return 1
        d=torch.distributions.Categorical(logits=lg); a=d.sample()
        self.ss.append(s); self.aa.append(a.item()); self.lp.append(d.log_prob(a).item()); self.vv.append(float(v)); return a.item()
    def store(self,r,done): self.rr.append(float(r) if np.isfinite(r) else 0.); self.dd.append(done)
    def update(self,lv=0.):
        if len(self.rr)<2: self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]; return
        vals=self.vv+[lv]; adv=[]; g=0.
        for t in reversed(range(len(self.rr))):
            m=1-self.dd[t]; delta=self.rr[t]+self.g*vals[t+1]*m-vals[t]
            g=delta+self.g*self.lam*m*g; adv.insert(0,g)
        adv=torch.FloatTensor(adv).to(DEVICE); adv=(adv-adv.mean())/(adv.std()+1e-8)
        ret=adv+torch.FloatTensor(self.vv).to(DEVICE)
        st=torch.FloatTensor(np.array(self.ss)).to(DEVICE)
        at=torch.LongTensor(self.aa).to(DEVICE); olp=torch.FloatTensor(self.lp).to(DEVICE)
        allp=list(self.bb.parameters())+list(self.ac.parameters())+list(self.cr.parameters())
        for _ in range(self.ep):
            lg,nv=self.fwd(st)
            if torch.isnan(lg).any(): break
            d=torch.distributions.Categorical(logits=lg); ratio=(d.log_prob(at)-olp).exp()
            al=-torch.min(ratio*adv,torch.clamp(ratio,1-self.eps,1+self.eps)*adv).mean()
            loss=al+self.vc*F.mse_loss(nv,ret)-self.ec*d.entropy().mean()
            if torch.isnan(loss): break
            self.opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(allp,.5); self.opt.step()
        self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]
    def greedy(self,s):
        self.bb.eval(); self.ac.eval()
        with torch.no_grad(): lg,_=self.fwd(torch.FloatTensor(safe(s)).to(DEVICE))
        self.bb.train(); self.ac.train()
        return 1 if torch.isnan(lg).any() else lg.argmax().item()
    def weights(self): return (copy.deepcopy(self.bb.state_dict()),copy.deepcopy(self.ac.state_dict()),copy.deepcopy(self.cr.state_dict()))
    def load(self,w): self.bb.load_state_dict(w[0]); self.ac.load_state_dict(w[1]); self.cr.load_state_dict(w[2])

# DQL
class DQL:
    def __init__(self,hid,nl,lr,gamma,buf,bs,tgt,eps_s,eps_e,eps_d,wd):
        self.g=gamma; self.bs=bs; self.tgt=tgt; self.steps=0
        self.eps=eps_s; self.eps_s=eps_s; self.eps_e=eps_e; self.eps_d=eps_d
        self.online=QNet(hid,nl).to(DEVICE); self.target=QNet(hid,nl).to(DEVICE)
        self.target.load_state_dict(self.online.state_dict()); self.target.eval()
        self.opt=torch.optim.Adam(self.online.parameters(),lr=lr,weight_decay=wd)
        self.buf=Buf(buf)
    def act(self,s):
        s=safe(s); self.eps=max(self.eps_e,self.eps_s-self.steps/self.eps_d*(self.eps_s-self.eps_e))
        if np.random.rand()<self.eps: return np.random.randint(3)
        self.online.eval()
        with torch.no_grad(): q=self.online(torch.FloatTensor(s).unsqueeze(0).to(DEVICE))
        self.online.train()
        return np.random.randint(3) if torch.isnan(q).any() else q.argmax().item()
    def store(self,s,a,r,ns,d): self.buf.push(safe(s),a,float(r) if np.isfinite(r) else 0.,safe(ns),d); self.steps+=1
    def update(self):
        if len(self.buf)<max(self.bs,2000): return
        s,a,r,ns,d,_=self.buf.sample(self.bs,DEVICE)
        with torch.no_grad():
            na=self.online(ns).argmax(1,keepdim=True); tq=self.target(ns).gather(1,na).squeeze(); tgt=r+self.g*tq*(1-d)
        cq=self.online(s).gather(1,a.unsqueeze(1)).squeeze(); loss=F.smooth_l1_loss(cq,tgt)
        if torch.isnan(loss): return
        self.opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(self.online.parameters(),.5); self.opt.step()
        if self.steps%self.tgt==0: self.target.load_state_dict(self.online.state_dict())
    def greedy(self,s):
        s=safe(s); self.online.eval()
        with torch.no_grad(): q=self.online(torch.FloatTensor(s).unsqueeze(0).to(DEVICE))
        self.online.train()
        return 1 if torch.isnan(q).any() else q.argmax().item()
    def weights(self): return copy.deepcopy(self.online.state_dict())
    def load(self,w): self.online.load_state_dict(w)

# DDPG
class DDPG:
    def __init__(self,hid,nl,la,lc,gamma,tau,buf,bs,ou_t,ou_s,wd):
        self.g=gamma; self.tau=tau; self.bs=bs
        self.actor=Actor(hid,nl).to(DEVICE); self.critic=Critic(hid,nl).to(DEVICE)
        self.at=Actor(hid,nl).to(DEVICE); self.ct=Critic(hid,nl).to(DEVICE)
        self.at.load_state_dict(self.actor.state_dict()); self.ct.load_state_dict(self.critic.state_dict())
        self.at.eval(); self.ct.eval()
        self.oa=torch.optim.Adam(self.actor.parameters(),lr=la,weight_decay=wd)
        self.oc=torch.optim.Adam(self.critic.parameters(),lr=lc,weight_decay=wd)
        self.buf=Buf(buf); self.noise=OU(ou_t,ou_s)
    def _soft(self,o,t):
        for p,pt in zip(o.parameters(),t.parameters()): pt.data.copy_(self.tau*p.data+(1-self.tau)*pt.data)
    def act(self,s,explore=True):
        s=safe(s); st=torch.FloatTensor(s).to(DEVICE)
        self.actor.eval(); disc,ca=self.actor.act(st); self.actor.train()
        if explore: ca=float(np.clip(ca+self.noise(),-1,1)); disc=0 if ca<-1/3 else (2 if ca>1/3 else 1)
        return disc,ca
    def store(self,s,a,r,ns,d,ca): self.buf.push(safe(s),a,float(r) if np.isfinite(r) else 0.,safe(ns),d,float(ca))
    def update(self):
        if len(self.buf)<max(self.bs,2000): return
        s,a,r,ns,d,ca=self.buf.sample(self.bs,DEVICE)
        with torch.no_grad():
            na=self.at(ns); tq=self.ct(ns,na).squeeze(); tgt=r+self.g*tq*(1-d)
        cq=self.critic(s,ca.unsqueeze(1)).squeeze(); cl=F.mse_loss(cq,tgt)
        if torch.isnan(cl): return
        self.oc.zero_grad(); cl.backward(); nn.utils.clip_grad_norm_(self.critic.parameters(),.5); self.oc.step()
        pa=self.actor(s); al=-self.critic(s,pa).mean()
        if torch.isnan(al): return
        self.oa.zero_grad(); al.backward(); nn.utils.clip_grad_norm_(self.actor.parameters(),.5); self.oa.step()
        self._soft(self.actor,self.at); self._soft(self.critic,self.ct)
    def greedy(self,s): disc,_=self.act(s,explore=False); return disc
    def weights(self): return (copy.deepcopy(self.actor.state_dict()),copy.deepcopy(self.critic.state_dict()))
    def load(self,w): self.actor.load_state_dict(w[0]); self.critic.load_state_dict(w[1])

print("Agents ready")

In [ ]:
def sharpe(vals):
    r=np.diff(np.array(vals))/np.array(vals[:-1])
    return float(np.mean(r)/(np.std(r)+1e-10)*np.sqrt(252)) if len(r)>1 else 0.

def eval_agent(ag,S,R):
    """Paper: select checkpoint by val Sharpe (no transaction cost in eval)."""
    env=make_env(S,R,rand=False,cost=EVAL_COST)
    obs,_=env.reset(); done=False
    while not done: obs,_,done,_,_=env.step(ag.greedy(obs))
    return sharpe(env.vals),env.metrics()

def make_agent(name,cfg):
    if name=="PG":   return PG(**cfg)
    elif name=="PPO": return PPO(**cfg)
    elif name=="DQL": return DQL(**cfg)
    elif name=="DDPG": return DDPG(**cfg)

def run_ep(ag,name,trS,trR):
    """Run one training episode."""
    env=make_env(trS,trR,rand=True,cost=True)
    obs,_=env.reset(); done=False; tot=0.; steps=0; lv=0.
    if name=="PG":
        while not done:
            a=ag.act(obs); obs,r,done,_,_=env.step(a); ag.store(r); tot+=r
        ag.update()
    elif name=="PPO":
        while not done:
            a=ag.act(obs); obs,r,done,_,_=env.step(a); ag.store(r,done); tot+=r
        if not done:
            with torch.no_grad(): _,lv=ag.fwd(torch.FloatTensor(safe(obs)).to(DEVICE)); lv=float(lv)
        ag.update(lv)
    elif name=="DQL":
        while not done and steps<500:
            a=ag.act(obs); no,r,done,_,_=env.step(a)
            ag.store(obs,a,r,no,done); ag.update(); obs=no; tot+=r; steps+=1
    elif name=="DDPG":
        ag.noise.reset()
        while not done and steps<500:
            a,ca=ag.act(obs); no,r,done,_,_=env.step(a)
            ag.store(obs,a,r,no,done,ca); ag.update(); obs=no; tot+=r; steps+=1
    return tot

def train(name,cfg,max_ep,trS,trR,vlS,vlR,label=""):
    ag=make_agent(name,cfg); best_sr=-np.inf; best_w=None; no_imp=0
    for ep in range(1,max_ep+1):
        try: run_ep(ag,name,trS,trR)
        except: continue
        if ep%EVAL_EVERY==0:
            try: sr,m=eval_agent(ag,vlS,vlR)
            except: sr=0.; m={"CR%":0,"SR":0,"MDD%":0}
            print(f"  [{name}{label}] ep={ep:4d} val_SR={sr:.3f} val_CR={m['CR%']:.1f}%")
            if sr>best_sr: best_sr=sr; best_w=ag.weights(); no_imp=0
            else:
                no_imp+=1
                if no_imp>=PATIENCE: print(f"  [{name}{label}] Early stop ep={ep}"); break
    if best_w: ag.load(best_w)
    return ag,best_sr

# Paper-style HP grid (3 configs per agent)
# Ranges informed by paper's Ray Tune search space
GRIDS = {
    "PG": [
        {"hid":128,"nl":2,"lr":1e-3,"gamma":0.99,"ent":0.01,"wd":1e-4},
        {"hid":256,"nl":2,"lr":5e-4,"gamma":0.99,"ent":0.05,"wd":1e-4},
        {"hid":128,"nl":3,"lr":2e-3,"gamma":0.99,"ent":0.01,"wd":1e-4},
    ],
    "PPO": [
        {"hid":128,"nl":2,"lr":3e-4,"gamma":0.99,"lam":0.95,"eps":0.2,"vc":0.5,"ec":0.01,"epochs":4,"wd":1e-4},
        {"hid":256,"nl":2,"lr":1e-4,"gamma":0.99,"lam":0.95,"eps":0.2,"vc":0.5,"ec":0.01,"epochs":8,"wd":1e-4},
        {"hid":128,"nl":2,"lr":3e-4,"gamma":0.99,"lam":0.98,"eps":0.1,"vc":0.5,"ec":0.05,"epochs":4,"wd":1e-4},
    ],
    "DQL": [
        {"hid":256,"nl":2,"lr":1e-3,"gamma":0.99,"buf":50000,"bs":64,"tgt":200,"eps_s":1.,"eps_e":0.05,"eps_d":10000,"wd":1e-4},
        {"hid":512,"nl":2,"lr":5e-4,"gamma":0.99,"buf":50000,"bs":64,"tgt":200,"eps_s":1.,"eps_e":0.05,"eps_d":10000,"wd":1e-4},
        {"hid":256,"nl":3,"lr":1e-3,"gamma":0.99,"buf":50000,"bs":128,"tgt":100,"eps_s":1.,"eps_e":0.05,"eps_d":8000,"wd":1e-4},
    ],
    "DDPG":[
        {"hid":256,"nl":2,"la":1e-4,"lc":1e-3,"gamma":0.99,"tau":0.005,"buf":50000,"bs":64,"ou_t":0.15,"ou_s":0.2,"wd":1e-4},
        {"hid":512,"nl":2,"la":5e-5,"lc":5e-4,"gamma":0.99,"tau":0.005,"buf":50000,"bs":64,"ou_t":0.15,"ou_s":0.2,"wd":1e-4},
        {"hid":256,"nl":2,"la":1e-4,"lc":1e-3,"gamma":0.99,"tau":0.01, "buf":50000,"bs":64,"ou_t":0.15,"ou_s":0.3,"wd":1e-4},
    ],
}
print("Training functions + HP grids ready")

In [ ]:
# ══════════════════════════════════════════════════════════════
# MAIN: Replicate paper for TSLA + BTC-USD
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

CLR={"B&H":"black","PG":"#1f77b4","PPO":"#ff7f0e","DQL":"#2ca02c","DDPG":"#d62728"}
final_results = {}

for ticker in ASSETS:
    print(f"\n{'='*65}")
    print(f"ASSET: {ticker}")
    print(f"{'='*65}")

    st = all_states[ticker]
    trS,trR,vlS,vlR,teS,teR = st["trS"],st["trR"],st["vlS"],st["vlR"],st["teS"],st["teR"]

    # Phase 1: HP search (simplified from paper's 180 Ray Tune trials)
    print(f"\nPhase 1: HP search (3 configs × {MAX_EP_HP} ep each)...")
    best_cfgs={}
    for name,grid in GRIDS.items():
        res=[]
        for i,cfg in enumerate(grid):
            ag,sr=train(name,cfg,MAX_EP_HP,trS,trR,vlS,vlR,f"-c{i+1}")
            res.append((sr,cfg)); del ag; gc.collect(); torch.cuda.empty_cache()
        res.sort(key=lambda x:x[0],reverse=True)
        best_cfgs[name]=res[0][1]
        print(f"  [{name}] best val_SR={res[0][0]:.3f}")

    # Phase 2: Final training with best HPs
    print(f"\nPhase 2: Final training (max {MAX_EP_FINAL} ep, paper-style early stopping)...")
    trained={}; val_srs={}
    for name,cfg in best_cfgs.items():
        ag,sr=train(name,cfg,MAX_EP_FINAL,trS,trR,vlS,vlR)
        trained[name]=ag; val_srs[name]=sr
        print(f"  [{name}] final val_SR={sr:.3f}")

    # Phase 3: Test evaluation (paper's official eval: Aug 2024+)
    bnh=make_env(teS,teR,rand=False,cost=False); bnh.reset()
    for _ in range(bnh.T-1): bnh.step(2)
    bnh_sr=sharpe(bnh.vals); bnh_m=bnh.metrics()

    rows=[{"Model":"B&H",**bnh_m,"ValSR":"--","vals":bnh.vals}]
    print(f"\n{'='*65}")
    print(f"RESULTS {ticker} — Test {TEST_START}+")
    print(f"{'='*65}")
    paper=PAPER_RESULTS[ticker]
    print(f"  {'B&H':6s} | CR={bnh_m['CR%']:7.1f}% SR={bnh_m['SR']:5.3f} MDD={bnh_m['MDD%']:5.1f}%  paper:{paper['B&H']}")

    for name,ag in trained.items():
        env=make_env(teS,teR,rand=False,cost=EVAL_COST)
        obs,_=env.reset(); done=False
        while not done: obs,_,done,_,_=env.step(ag.greedy(obs))
        sr=sharpe(env.vals); m=env.metrics(); beat=m["CR%"]>bnh_m["CR%"]
        rows.append({"Model":name,**m,"ValSR":round(val_srs[name],3),"vals":env.vals})
        beat_str="BEAT" if beat else "----"
        print(f"  {name:6s} | CR={m['CR%']:7.1f}% SR={m['SR']:5.3f} MDD={m['MDD%']:5.1f}%  [{beat_str}]"
              f"  paper:{paper.get(name,'--'):8s}  ValSR={val_srs[name]:.3f}"
              f"  long={m['pct_long']:.0f}% flat={m['pct_flat']:.0f}% short={m['pct_short']:.0f}%")

    final_results[ticker]=rows
    df=pd.DataFrame([{k:v for k,v in r.items() if k!="vals"} for r in rows])
    df.to_csv(f"/kaggle/working/results_{ticker}_paper_replication.csv",index=False)

In [ ]:
# ══════════════════════════════════════════════════════════════
# SUMMARY + VISUALIZATION
# ══════════════════════════════════════════════════════════════
print("\n"+"="*70)
print("FINAL COMPARISON VS PAPER (CLaC@FinMMEval2026)")
print("="*70)
print(f"{'Asset':<10} {'Model':<8} {'Paper CR':>10} {'Ours CR':>10} {'Our SR':>8} {'Our MDD':>8}")
print("-"*60)
for ticker,rows in final_results.items():
    paper=PAPER_RESULTS[ticker]
    for r in rows:
        m=r["Model"]; pap=paper.get(m,"--")
        cr_diff=""
        if m!="B&H" and pap!="--":
            try:
                diff=r["CR%"]-float(pap.replace("%",""))
                cr_diff=f" ({'+' if diff>=0 else ''}{diff:.1f}%)"
            except: pass
        print(f"{ticker:<10} {m:<8} {pap:>10} {r['CR%']:>10.1f}%{cr_diff}  {r['SR']:>6.3f}  {r['MDD%']:>6.1f}%")

# Visualization: side-by-side TSLA + BTC
fig,axes=plt.subplots(1,2,figsize=(18,7))
for idx,(ticker,rows) in enumerate(final_results.items()):
    ax=axes[idx]
    for r in rows:
        m=r["Model"]; vals=r["vals"]
        ls="--" if m=="B&H" else "-"; lw=2.5 if m=="B&H" else 1.5
        pap=PAPER_RESULTS[ticker].get(m,"--")
        ax.plot(vals,label=f"{m}: {r['CR%']:.1f}% (paper:{pap})",
                color=CLR.get(m,"gray"),ls=ls,lw=lw)
    ax.set_xlabel("Trading Days",fontsize=11)
    ax.set_ylabel("Portfolio Value ($)",fontsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f"${x/1e3:.0f}K"))
    bnh_cr=next(r["CR%"] for r in rows if r["Model"]=="B&H")
    ax.set_title(f"{ticker} — Test {TEST_START}+ | B&H={bnh_cr:.1f}%",fontsize=13,fontweight="bold")
    ax.legend(fontsize=8,loc="upper left"); ax.grid(True,alpha=0.3)

fig.suptitle("CLaC@FinMMEval2026 Replication\n"
             "LLaMA 3.2 1B zero-shot → Qwen2.5-3B fine-tuned (Acc=90.42%, F1=0.9047)\n"
             f"Train: up to {TRAIN_END} | Val: {VAL_START}–{VAL_END} | Test: {TEST_START}+",
             fontsize=12,fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/paper_replication.png",dpi=150,bbox_inches="tight")
plt.close()
print("\nSaved: paper_replication.png")
print("Saved: results_TSLA_paper_replication.csv")
print("Saved: results_BTC-USD_paper_replication.csv")